# Transformations and Differencing Pipeline

**Week 2 — Stationarity | Notebook 3 of 3**

This notebook demonstrates the **complete pipeline** for making a non-stationary series stationary:

1. **Variance stabilisation** (log or Box-Cox) — always first
2. **Differencing** (first and/or seasonal) — to remove trend and seasonality
3. **Verification** — ADF + KPSS + rolling plots on the result

We also demonstrate **over-differencing** and how to detect it.

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import acf

from week__01__fundamentals.ts_loader import TimeSeriesLoader
from week__02__stationarity.stationarity_tests import StationarityTester
from week__02__stationarity.transformations import VarianceStabiliser, Differencer

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 4)

tester = StationarityTester()

## 1. Load Airline Passengers

In [ ]:
airline, _ = TimeSeriesLoader("AirPassengers").from_statsmodels("airline")
print(f"Series length: {len(airline)}, Range: {airline.index[0].date()} to {airline.index[-1].date()}")

raw_result = tester.fit(airline)
print(f"\nRaw series: {raw_result.verdict.name}")
print(raw_result)

## 2. Step 1 — Variance Stabilisation

The Airline Passengers series has variance that grows with its level (seasonal swings
get wider as passenger numbers increase). We **must** stabilise variance before differencing.

### 2a. Auto-select via Box-Cox

In [ ]:
stabiliser_auto = VarianceStabiliser()  # auto-select
airline_transformed = stabiliser_auto.fit_transform(airline)

print(f"Auto-selected transform: {stabiliser_auto.result}")
print(f"Box-Cox λ was close to 0 → log transform chosen (simpler & more interpretable)")

In [ ]:
# Compare: explicit Box-Cox with auto lambda
from scipy.stats import boxcox

_, lam = boxcox(airline.values)
print(f"Box-Cox auto λ = {lam:.4f}")
print(f"λ ≈ 0 confirms log is the right choice (log is Box-Cox with λ=0)")

### 2b. Compare Raw vs Log-Transformed

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))

# Raw
axes[0, 0].plot(airline, color="steelblue")
axes[0, 0].set_title("Raw Series")

diag_raw = StationarityTester.rolling_diagnostics(airline, window=12)
axes[1, 0].plot(diag_raw["rolling_var"], color="darkorange")
axes[1, 0].set_title("Raw — Rolling Variance (growing!)")

# Log-transformed
axes[0, 1].plot(airline_transformed, color="seagreen")
axes[0, 1].set_title("Log-Transformed Series")

diag_log = StationarityTester.rolling_diagnostics(airline_transformed, window=12)
axes[1, 1].plot(diag_log["rolling_var"], color="darkorange")
axes[1, 1].set_title("Log — Rolling Variance (stabilised!)")

plt.tight_layout()
plt.show()

**Interpretation:** After log transform, the rolling variance is much more stable. The seasonal amplitude
no longer grows with the level. Now we can safely difference.

## 3. Step 2 — Differencing

### 3a. First Differencing (d=1)

First differencing removes linear trend: `y'ₜ = yₜ - yₜ₋₁`

In [ ]:
diff = Differencer(seasonal_period=12)
airline_d1 = diff.difference(airline_transformed, d=1, D=0)

print(f"After d=1: {len(airline_d1)} observations")
result_d1 = tester.fit(airline_d1)
print(result_d1)

### 3b. Seasonal Differencing (D=1, m=12)

Seasonal differencing removes the annual pattern: `y'ₜ = yₜ - yₜ₋₁₂`

In [ ]:
airline_D1 = diff.difference(airline_transformed, d=0, D=1, m=12)

print(f"After D=1: {len(airline_D1)} observations")
result_D1 = tester.fit(airline_D1)
print(result_D1)

### 3c. Both: d=1 + D=1 (the standard ARIMA preprocessing)

In [ ]:
airline_d1_D1 = diff.difference(airline_transformed, d=1, D=1, m=12)

print(f"After d=1, D=1: {len(airline_d1_D1)} observations")
result_d1_D1 = tester.fit(airline_d1_D1)
print(result_d1_D1)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))

panels = [
    (airline_transformed, "Log(Airline) — still non-stationary"),
    (airline_d1, "d=1 — trend removed"),
    (airline_D1, "D=1 (m=12) — seasonality removed"),
    (airline_d1_D1, "d=1, D=1 — both removed (stationary!)"),
]

for ax, (data, title) in zip(axes.flat, panels):
    ax.plot(data, linewidth=0.8)
    ax.set_title(title)
    ax.axhline(y=data.mean(), color="red", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

### 3d. Before/After Rolling Statistics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 6))

# Before (raw)
diag_before = StationarityTester.rolling_diagnostics(airline, window=12)
axes[0, 0].plot(diag_before["rolling_mean"], color="red")
axes[0, 0].set_title("BEFORE — Rolling Mean (trending)")
axes[1, 0].plot(diag_before["rolling_var"], color="darkorange")
axes[1, 0].set_title("BEFORE — Rolling Variance (growing)")

# After (log + d=1 + D=1)
diag_after = StationarityTester.rolling_diagnostics(airline_d1_D1, window=12)
axes[0, 1].plot(diag_after["rolling_mean"], color="red")
axes[0, 1].set_title("AFTER — Rolling Mean (flat!)")
axes[1, 1].plot(diag_after["rolling_var"], color="darkorange")
axes[1, 1].set_title("AFTER — Rolling Variance (stable!)")

plt.tight_layout()
plt.show()

## 4. Auto-Detection with `Differencer.fit_transform()`

Instead of manually choosing d and D, let the `Differencer` auto-detect using ADF + KPSS in a loop.

In [ ]:
auto_diff = Differencer(max_d=2, max_D=1, seasonal_period=12)
airline_auto = auto_diff.fit_transform(airline_transformed)

print(f"Auto-detected: {auto_diff.result}")
print(f"\nVerification:")
auto_result = tester.fit(airline_auto)
print(auto_result)

## 5. Over-Differencing Demo

**What happens if we difference too much?**

If a series only needs d=1 but we apply d=2, the result will have **artificially negative
autocorrelation at lag 1** (ACF₁ ≈ -0.5). This is the diagnostic for over-differencing.

We demonstrate on a synthetic random walk (needs exactly d=1).

In [ ]:
rng = np.random.default_rng(seed=42)
dates = pd.date_range("2000-01-01", periods=500, freq="D")
random_walk = pd.Series(np.cumsum(rng.standard_normal(500)), index=dates, name="RandomWalk")

diff_rw = Differencer(seasonal_period=7)

# Correct: d=1
rw_d1 = diff_rw.difference(random_walk, d=1, D=0)
acf_d1 = acf(rw_d1.dropna(), nlags=5, fft=True)

# Over-differenced: d=2
rw_d2 = diff_rw.difference(random_walk, d=2, D=0)
acf_d2 = acf(rw_d2.dropna(), nlags=5, fft=True)

print("d=1 (correct):")
print(f"  ACF(1) = {acf_d1[1]:.4f}  ← near zero (white noise, as expected)")
print(f"  Over-differenced? {Differencer.is_over_differenced(rw_d1)}")

print(f"\nd=2 (over-differenced):")
print(f"  ACF(1) = {acf_d2[1]:.4f}  ← strongly negative (diagnostic signal!)")
print(f"  Over-differenced? {Differencer.is_over_differenced(rw_d2)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

axes[0].bar(range(6), acf_d1[:6], color="steelblue")
axes[0].axhline(y=-0.5, color="red", linestyle="--", alpha=0.5, label="Over-diff threshold")
axes[0].set_title("d=1 (correct) — ACF")
axes[0].set_xlabel("Lag")
axes[0].legend()

axes[1].bar(range(6), acf_d2[:6], color="indianred")
axes[1].axhline(y=-0.5, color="red", linestyle="--", alpha=0.5, label="Over-diff threshold")
axes[1].set_title("d=2 (over-differenced!) — ACF")
axes[1].set_xlabel("Lag")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Complete Pipeline Summary

The full stationarity pipeline for Airline Passengers:

```
Raw series (non-stationary: trend + growing variance + seasonality)
    │
    ▼ Step 1: Log transform (variance stabilisation)
    │
    ▼ Step 2: Seasonal differencing D=1, m=12 (remove annual pattern)
    │
    ▼ Step 3: First differencing d=1 (remove remaining trend)
    │
    ▼ Step 4: Verify with ADF + KPSS → STATIONARY ✓
    │
    ▼ Ready for ARIMA(p,1,q)(P,1,Q)[12] model fitting (Week 3–4)
```

**Key rules:**
1. Always stabilise variance **before** differencing
2. Use both ADF and KPSS together — never rely on one test alone
3. Check for over-differencing: ACF(1) < -0.5 means you went too far
4. The d and D values determined here feed directly into the ARIMA model order